## Test the /search endpoints for CADIP products
### Test Cases:
#### 1. "search" - using the /cadip/search endpoint
#### 2. "filter" - using the /cadip/collections/collection_id/items endpoint
#### 3. "filter" - pairs of properties - using the /cadip/collections/collection_id/items endpoint
#### 4. "sort" with /cadip/search endpoint
#### 5. "sort" with /cadip/collections/collection_id/items endpoint

In [ ]:
from pystac_client.exceptions import APIError
# Init environment before running a demo notebook.
from resources.utils import *  
init_demo()
# Reload the global vars again
from resources.utils import * 

# Counters for total requests and failures
total_requests = 0
total_failures = 0
failed_requests = []

cadip_search_properties = {
    "cadip:acquisition_id": "53186_A1",
    "cadip:antenna_id": "MSP21",
    "cadip:antenna_status_ok": "true",
    "cadip:delivery_push_ok": "true",
    "cadip:downlink_status_ok": "true",
    "cadip:front_end_id": "01",
    "cadip:front_end_status_ok": "true",
    "cadip:num_channels": "2",
    "cadip:planned_data_start": "2020-01-05T07:22:04.051Z",
    "cadip:planned_data_stop": "2020-01-05T07:31:04.051Z",
    "cadip:retransfer": "false",
    "cadip:station_unit_id": "01",
    "constellation": "sentinel-1",
    "datetime": "2020-01-05T18:52:26.165Z",
    "end_datetime": "2020-01-05T07:42:04.051Z",
    "id": "S1A_20200105072204051312",
    "platform": "sentinel-1a",
    "published": "2020-01-05T18:52:26.165Z",
    "sat:absolute_orbit": "53186",
    "start_datetime": "2020-01-05T07:22:04.051Z"
}

cadip_sortable_properties = [
    "start_datetime", "datetime", "end_datetime", "published", "platform", "cadip:id",
    "cadip:num_channels", "cadip:station_unit_id", "sat:absolute_orbit", "cadip:acquisition_id",
    "cadip:antenna_id", "cadip:front_end_id", "cadip:retransfer", "cadip:antenna_status_ok",
    "cadip:front_end_status_ok", "cadip:planned_data_start", "cadip:planned_data_stop",
    "cadip:downlink_status_ok", "cadip:delivery_push_ok"
]

cadip_collection = "cadip_sentinel1"

In [ ]:
# 1. "search" - using the /cadip/search endpoint

for prop, value in cadip_search_properties.items():    
    filter=f"{prop}='{value}'"    
    print(f"filter: {filter}")
    total_requests += 1
    try:
        items_collection_cadip = cadip_client.search(method="GET", stac_filter=filter)
        total_requests += 1        
        print(f"✅ RsClient request succeded for {prop} ({value}): {len(items_collection_cadip)} items")
    except RuntimeError as e:
        print(f"❌ RsClient failed for {prop}: {e}")
        total_failures += 1
        failed_requests.append(str(e))
    print("-" * 80)

In [ ]:
# 2. "filter" - using the /cadip/search endpoint but now the collection name will be present in the filter

for prop, value in cadip_search_properties.items():
    filter=f"{prop}='{value}'"    
    print(f"filter: {filter}")
    total_requests += 1
    
    try:
        items_collection_cadip = cadip_client.search(method="GET", 
                                                    collections = [cadip_collection],
                                                    stac_filter=filter)        
        print(f"✅ RsClient request succeded for {prop} ({value}): {len(items_collection_cadip)} items") 
    except APIError as e:
        print(f"❌ RsClient failed for {prop}: {e}")
        total_failures += 1
        failed_requests.append(str(e))
    print("-" * 80)

In [ ]:
# 3. "filter" by pairs of properties - using the /cadip/collections/<id>/items endpoint

# Generate all possible pairs of search properties
keys = list(cadip_search_properties.keys())
for i in range(len(keys)):
    for j in range(i + 1, len(keys)):
        prop1, prop2 = keys[i], keys[j]
        value1, value2 = cadip_search_properties[prop1], cadip_search_properties[prop2]

        filter=f"{prop1}='{value1}' AND {prop2}='{value2}'"
        print(f"filter = {filter}")        
        total_requests += 1

        try:
            items_collection_cadip = cadip_client.search(method="GET", 
                                                    collections = [cadip_collection],
                                                    stac_filter=filter)        
            print(f"✅ RsClient request succeded for {prop} ({value}): {len(items_collection_cadip)} items")            
        except RuntimeError as e:
            print(f"❌ Request failed for {prop1} and {prop2}: {e}")
            total_failures += 1
            failed_requests.append(str(e))
        print("-" * 80)

In [ ]:
# 4. "sort" with "search" /cadip/search endpoint": requests for each search property with sorting and response limit

# for prop, value in cadip_search_properties.items():
#     formatted_value = f'"{value}"'
#     for sort_prop in cadip_sortable_properties:
#         url = f"{BASE_URL_SEARCH}?filter={prop}={formatted_value}&sortby=-{sort_prop}&limit=5"
#         print(f"Requesting: {url}")
#         total_requests += 1
        
#         try:
#             response = http_session.get(url)
#             response.raise_for_status()
#             data = response.json()
            
#             # Count how many items are in "features"
#             num_items = len(data.get("features", []))
#             print(f"✅ Request succeded for  {prop} ({value}) sorted by {sort_prop}: {num_items} items")
#         except requests.exceptions.RequestException as e:
#             print(f"❌ Request failed for {prop} sorted by {sort_prop}: {e}")
#             total_failures += 1
#             failed_requests.append(str(e))
#         print("-" * 80)

In [ ]:
# 5. "filter" with "search" /cadip/collections/collection_id/items endpoint": requests for each search property with sorting and response limit

# for prop, value in cadip_search_properties.items():
#     formatted_value = f'"{value}"'
#     for sort_prop in cadip_sortable_properties:
#         url = f"{BASE_URL_FILTER}?filter={prop}={formatted_value}&sortby=-{sort_prop}&limit=5"
#         print(f"Requesting: {url}")
#         total_requests += 1
        
#         try:
#             response = http_session.get(url)
#             response.raise_for_status()
#             data = response.json()
            
#             # Count how many items are in "features"
#             num_items = len(data.get("features", []))
#             print(f"✅ Request succeded for  {prop} ({value}) sorted by {sort_prop}: {num_items} items")
#         except requests.exceptions.RequestException as e:
#             print(f"❌ Request failed for {prop} sorted by {sort_prop}: {e}")
#             total_failures += 1
#             failed_requests.append(str(e))
#         print("-" * 80)

In [ ]:
# ****************************** Summary report ******************************
print("*" * 30, "Summary report", "*" * 30)
print(f"Total requests: {total_requests}")
print(f"Total failed requests: {total_failures}")
if failed_requests:
    print("Failed requests details:")
    for error in failed_requests:
        print(error)

assert total_failures == 0
assert not failed_requests